# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [5]:
import warnings

from openpyxl.worksheet import dimensions

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  evenements_volumineux.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [6]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (16992, 47)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [7]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5162,17445914.41,3379.68,1167.38,26602
Audio,3949,9234318.00,2338.39,1073.00,15078
Mobilier,3936,8363145.58,2124.78,717.48,18274
Accessoire,3870,7575740.68,1957.56,834.65,14978


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [8]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie   montant  ca_categorie  part_dans_categorie
0  Informatique   457.881  1.744591e+07               0.0026
1      Mobilier   140.960  8.363146e+06               0.0017
2         Audio  2073.360  9.234318e+06               0.0225
3  Informatique  2281.000  1.744591e+07               0.0131
4      Mobilier  1904.490  8.363146e+06               0.0228

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [9]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,5738,P0027,0.165
telephone,2865,P0015,0.166
web,8389,P0038,0.170


### Partie autonome

In [10]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.

ca_table = sales.pivot_table(
    index='categorie',
    columns='canal',
    values='montant',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
    fill_value=0,
)

display(ca_table.round(2))

canal,boutique,telephone,web,Total
categorie,,,,
Accessoire,2168517.40,2311009.93,3096213.34,7575740.68
Audio,2826937.74,861984.95,5545395.31,9234318.00
Informatique,6269923.45,4832327.05,6343663.91,17445914.41
Mobilier,3379260.02,2469037.16,2514848.40,8363145.58
Total,14644638.61,10474359.09,17500120.96,42619118.67


In [11]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.

by_customer = sales.groupby('id_client').agg(
    nb_commandes=('id_commande', 'nunique'),
    ca_total=('montant', 'sum'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique'),
)

# Montant moyen par commande.
by_customer['panier_moyen'] = (
    by_customer['ca_total'] / by_customer['nb_commandes']
)

# Classer les clients du plus grand CA au plus petit.
top_customers = (
    by_customer
    .sort_values('ca_total', ascending=False)
    .head(10)
)

display(top_customers.round(2))


,nb_commandes,ca_total,premiere_commande,derniere_commande,nb_categories,panier_moyen
id_client,,,,,,
C00696,7,1110180.19,2023-03-13 00:00:00,2024-11-19 00:00:00,3,158597.17
C02327,4,907834.48,2023-10-31 00:00:00,2024-12-28 14:36:00,2,226958.62
C00279,10,876999.45,2023-01-24 00:00:00,2024-11-05 00:00:00,4,87699.95
C02631,10,788266.88,2023-01-12 00:00:00,2024-12-19 00:00:00,4,78826.69
C00941,7,783179.31,2023-01-29 00:00:00,2024-09-15 00:00:00,3,111882.76
C00355,13,779167.47,2023-04-09 16:29:00,2024-09-25 00:00:00,4,59935.96
C02101,6,750743.90,2023-01-05 11:00:00,2024-02-03 00:00:00,3,125123.98
C00806,5,707386.06,2023-05-03 00:00:00,2024-08-26 00:00:00,2,141477.21
C02285,7,687164.51,2023-01-10 10:43:00,2024-12-27 00:00:00,2,98166.36


In [12]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?


# Calculer la médiane de chaque catégorie et la reporter sur chaque ligne.
median_by_category = (
    sales.groupby('categorie')['montant'].transform('median')
)

# Signaler les ventes dépassant 150 % de cette médiane.
sales['montant_eleve'] = (
    sales['montant'] > median_by_category * 1.5
)

# Calculer la proportion de lignes signalées.
share = sales['montant_eleve'].mean() * 100

print(f"Part des ventes signalées : {share:.2f} %")

display(sales[[
    'categorie',
    'montant',
    'montant_eleve',
]].head(10))


Part des ventes signalées : 33.32 %


,categorie,montant,montant_eleve
0,Informatique,457.8810,False
1,Mobilier,140.9600,False
2,Audio,2073.3600,True
3,Informatique,2281.0000,True
4,Mobilier,1904.4900,True
5,Audio,1419.0400,False
6,Mobilier,400.0525,False
7,Informatique,1040.5870,False
8,Accessoire,2570.7000,True
9,Audio,608.7800,False


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [13]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 16992
Après jointure produits : 16992
Lignes sans produit correspondant : 75 (0.44%)
Identifiants fautifs : <ArrowStringArray>
['P9999']
Length: 1, dtype: str


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [14]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
16992 lignes -> 17287 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [15]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_client
   C00001
   C00002
   C00003
   C00004
   C00005 ...


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [ ]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).

df_analytique = sales.copy()
n_initial = len(df_analytique)

# Journal : relever les nombres réels après chaque jointure.
assembly_counts = [{
    'etape': 'Ventes de départ',
    'lignes_avant': n_initial,
    'lignes_apres': n_initial,
    'ecart': 0,
    'lignes_sans_correspondance': 0,
    'explication': 'Point de départ : une ligne par ligne de vente conservée.',
}]

# Signaler les ventes sans correspondance dans les référentiels.
df_analytique['produit_inconnu'] = (
    ~df_analytique['id_produit'].isin(products['id_produit'])
)

df_analytique['client_inconnu'] = (
    ~df_analytique['id_client'].isin(customers['id_client'])
)

df_analytique['magasin_inconnu'] = (
    ~df_analytique['id_magasin'].isin(stores['id_magasin'])
)

print("Lignes au départ :", n_initial)

# 1. Ajouter les informations produits.
df_analytique = df_analytique.merge(
    products,
    on='id_produit',
    how='left',
    validate='many_to_one',
    suffixes=('', '_prod'),
)

print("Après produits :", len(df_analytique))

assembly_counts.append({
    'etape': 'Jointure produits (id_produit)',
    'lignes_avant': assembly_counts[-1]['lignes_apres'],
    'lignes_apres': len(df_analytique),
    'ecart': len(df_analytique) - assembly_counts[-1]['lignes_apres'],
    'lignes_sans_correspondance': int(df_analytique['produit_inconnu'].sum()),
    'explication': 'Jointure gauche : ventes conservées. Clé unique à droite, contrôlée par validate : aucune multiplication.',
})
assert len(df_analytique) == n_initial, 'La jointure a changé le nombre de lignes'



# 2. Ajouter les informations clients.
df_analytique = df_analytique.merge(
    customers,
    on='id_client',
    how='left',
    validate='many_to_one',
    suffixes=('', '_client'),
)

print("Après clients :", len(df_analytique))

assembly_counts.append({
    'etape': 'Jointure clients (id_client)',
    'lignes_avant': assembly_counts[-1]['lignes_apres'],
    'lignes_apres': len(df_analytique),
    'ecart': len(df_analytique) - assembly_counts[-1]['lignes_apres'],
    'lignes_sans_correspondance': int(df_analytique['client_inconnu'].sum()),
    'explication': 'Jointure gauche : ventes conservées. Clé unique à droite, contrôlée par validate : aucune multiplication.',
})
assert len(df_analytique) == n_initial, 'La jointure a changé le nombre de lignes'



# 3. Ajouter les informations magasins.
df_analytique = df_analytique.merge(
    stores,
    on='id_magasin',
    how='left',
    validate='many_to_one',
    suffixes=('', '_magasin'),
)

print("Après magasins :", len(df_analytique))

assembly_counts.append({
    'etape': 'Jointure magasins (id_magasin)',
    'lignes_avant': assembly_counts[-1]['lignes_apres'],
    'lignes_apres': len(df_analytique),
    'ecart': len(df_analytique) - assembly_counts[-1]['lignes_apres'],
    'lignes_sans_correspondance': int(df_analytique['magasin_inconnu'].sum()),
    'explication': 'Jointure gauche : ventes conservées. Clé unique à droite, contrôlée par validate : aucune multiplication.',
})
assert len(df_analytique) == n_initial, 'La jointure a changé le nombre de lignes'


# Compter les lignes de vente concernées par les clés orphelines.
print("Ventes sans produit connu :", df_analytique['produit_inconnu'].sum())
print("Ventes sans client connu :", df_analytique['client_inconnu'].sum())
print("Ventes sans magasin connu :", df_analytique['magasin_inconnu'].sum())

display(df_analytique.head())

# Choix : conserver les ventes sans correspondance pour ne pas perdre
# leurs montants. Les informations non retrouvées restent manquantes,
# et les colonnes *_inconnu permettent de repérer les lignes concernées.


Lignes au départ : 16992
Après produits : 16992
Après clients : 16992
Après magasins : 16992
Ventes sans produit connu : 75
Ventes sans client connu : 0
Ventes sans magasin connu : 0


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison,montant,libelle,categorie,sous_categorie,cout_achat,montant_apres_remise,panier_moyen_client,commande_non_aboutie,ecart_prix_median_categorie,...,sous_categorie_regroupee=Micro,sous_categorie_regroupee=Ordinateur portable,sous_categorie_regroupee=Sacoche,sous_categorie_regroupee=Souris,montant_zscore,produit_inconnu,client_inconnu,magasin_inconnu,libelle_prod,categorie_prod,sous_categorie_prod,cout_achat_prod,date_inscription,segment,ville,age,nom_magasin,ville_magasin,region,surface_m2
0,CMD0003112,2024-01-18 19:53:00,C02300,P0009,M05,3,160.66,5.0,telephone,retourne,lille,457.881,Clavier modele 3,Informatique,Clavier,86.41,457.881,1478.870333,1,-702.15,...,0,0,0,0,-0.076478,False,False,False,Clavier modele 3,Informatique,Clavier,86.41,2021-04-18,Particulier,PARIS,39.0,Boutique Bordeaux,Bordeaux,Nouvelle-Aquitaine,769
1,CMD0000107,2024-01-08 00:00:00,C02791,P0028,M03,2,70.48,0.0,boutique,livre,toulouse,140.960,Lampe modele 1,Mobilier,Lampe,34.81,140.960,860.969571,0,-382.75,...,0,0,0,0,-0.088267,False,False,False,Lampe modele 1,Mobilier,Lampe,34.81,2024-02-19,Particulier,PARIS,36.0,Boutique Marseille,Marseille,PACA,460
2,CMD0010050,2023-04-15 00:00:00,990031,P0017,M04,3,691.12,0.0,boutique,livre,marseille,2073.360,Enceinte modele 2,Audio,Enceinte,347.41,2073.360,1170.664800,0,-37.75,...,0,0,0,0,-0.016386,False,False,False,Enceinte modele 2,Audio,Enceinte,347.41,2019-02-24,Professionnel,Bordeaux,NaN,Boutique Lille,Lille,Hauts-de-France,245
3,CMD0001979,2024-11-06 00:00:00,C01883,P0010,M04,4,570.25,0.0,web,retourne,marseille,2281.000,Souris modele 1,Informatique,Souris,283.64,2281.000,1225.822375,1,-292.56,...,0,0,0,1,-0.008663,False,False,False,Souris modele 1,Informatique,Souris,283.64,2019-06-01,Particulier,Nantes,77.0,Boutique Lille,Lille,Hauts-de-France,245
4,CMD0010513,2024-11-02 00:00:00,C00571,P0024,M02,2,1058.05,10.0,web,livre,marseille,1904.490,Bureau modele 3,Mobilier,Bureau,554.59,1904.490,1392.776667,0,604.82,...,0,0,0,0,-0.022668,False,False,False,Bureau modele 3,Mobilier,Bureau,554.59,2021-01-02,Particulier,Strasbourg,44.0,Boutique Lyon,Lyon,Auvergne-Rhone-Alpes,853


### Bilan des lignes à chaque étape de l’assemblage

Le tableau suivant est calculé pendant Q4. **Écart = lignes après − lignes avant**.
La colonne « lignes sans correspondance » compte les lignes de vente concernées,
ce n’est pas le nombre d’identifiants distincts et ce n’est pas une suppression.


In [ ]:
bilan_lignes = pd.DataFrame(assembly_counts)
display(bilan_lignes)

print('Bilan :', len(sales), 'lignes en entrée et', len(df_analytique), 'lignes en sortie.')
print('Écart total :', len(df_analytique) - len(sales))


Bilan : 16992 lignes en entrée et 16992 lignes en sortie.
Écart total : 0


,etape,lignes_avant,lignes_apres,ecart,lignes_sans_correspondance,explication
0,Ventes de départ,16992,16992,0,0,Point de départ : une ligne par ligne de vente...
1,Jointure produits (id_produit),16992,16992,0,75,Jointure gauche : ventes conservées. Clé uniqu...
2,Jointure clients (id_client),16992,16992,0,0,Jointure gauche : ventes conservées. Clé uniqu...
3,Jointure magasins (id_magasin),16992,16992,0,0,Jointure gauche : ventes conservées. Clé uniqu...


**Explication des résultats observés**

| Étape | Avant | Après | Écart | Lignes sans correspondance |
|---|---:|---:|---:|---:|
| Ventes de départ | 16 992 | 16 992 | 0 | — |
| Ajout des produits | 16 992 | 16 992 | 0 | 75 |
| Ajout des clients | 16 992 | 16 992 | 0 | 0 |
| Ajout des magasins | 16 992 | 16 992 | 0 | 0 |

- **Pourquoi aucun écart de lignes ?** Chaque jointure est un `left join` : aucune vente n’est supprimée. `validate='many_to_one'` garantit au plus une correspondance à droite : aucune vente n’est multipliée.
- **Pourquoi 75 ventes sans produit mais aucune ligne perdue ?** Ces ventes référencent `P9999`, absent de `products`. Elles sont conservées et signalées par `produit_inconnu=True`. Les attributs du référentiel produit ajouté ne sont pas renseignés pour ces lignes. Cela représente environ **0,44 %** des ventes, pour **un seul identifiant distinct**.
- **Clients et magasins :** toutes les clés de vente trouvent une correspondance.
- **Limite :** conserver le nombre de lignes ne suffit pas à garantir l’intégrité référentielle. Les 75 références produit doivent encore être traitées lors de la construction du modèle final, par exemple avec une ligne de dimension explicitement inconnue, sans inventer ses attributs.

Les chiffres ci-dessus correspondent aux données actuelles ; le tableau Python est recalculé à chaque exécution de Q4.


In [17]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.

def safe_merge(left, right, **kwargs):
    n_before = len(left)

    # Ajouter une colonne indiquant si la correspondance a été trouvée.
    result = left.merge(right, indicator=True, **kwargs)

    # Pour un left join, vérifier que le nombre de lignes est conservé.
    if kwargs.get('how', 'inner') == 'left':
        if len(result) != n_before:
            raise ValueError(
                f"La jointure a changé le nombre de lignes : "
                f"{n_before} → {len(result)}"
            )

    unmatched = result['_merge'] == 'left_only'

    if n_before > 0:
        unmatched_rate = unmatched.sum() / n_before
    else:
        unmatched_rate = 0

    print(f"Lignes de gauche sans correspondance : {unmatched_rate:.2%}")

    # Retirer la colonne technique avant de renvoyer le résultat.
    return result.drop(columns='_merge')


---
## Atelier 6.3 — Formats long et large (35 min)

In [18]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{month:02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,367032.0,296781.0,333273.0,1218847.0,539612.0,339807.0,334524.0,1204402.0,491953.0,1090514.0,893844.0,465151.0
Audio,1180732.0,658118.0,389644.0,1082484.0,427252.0,1169529.0,575026.0,1057970.0,418011.0,431277.0,1169598.0,674677.0
Informatique,1808515.0,1586185.0,2721960.0,676080.0,2323577.0,750591.0,2923108.0,1046418.0,565554.0,596944.0,773439.0,1673544.0
Mobilier,849867.0,328436.0,369598.0,1133552.0,1356966.0,873502.0,361750.0,195968.0,1405982.0,300714.0,715121.0,471688.0


In [19]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,367032.0
1,Audio,mois_01,1180732.0
2,Informatique,mois_01,1808515.0
3,Mobilier,mois_01,849867.0
4,Accessoire,mois_02,296781.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [20]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.

widew = long.pivot(
    index='categorie',
    columns='mois',
    values='ca',
)
display(widew)

mois,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,367032.0,296781.0,333273.0,1218847.0,539612.0,339807.0,334524.0,1204402.0,491953.0,1090514.0,893844.0,465151.0
Audio,1180732.0,658118.0,389644.0,1082484.0,427252.0,1169529.0,575026.0,1057970.0,418011.0,431277.0,1169598.0,674677.0
Informatique,1808515.0,1586185.0,2721960.0,676080.0,2323577.0,750591.0,2923108.0,1046418.0,565554.0,596944.0,773439.0,1673544.0
Mobilier,849867.0,328436.0,369598.0,1133552.0,1356966.0,873502.0,361750.0,195968.0,1405982.0,300714.0,715121.0,471688.0


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [21]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [28]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.

fact_sales =  pd.DataFrame({
    "cle_date": sales['date_commande'],
    "id_client": sales['id_client'],
    "id_produit": sales['id_produit'],
    "id_magasin": sales['id_magasin'],
    "quantite": sales['quantite'],
    "prix_unitaire" : sales['prix_unitaire'],
    "remise_pct" : sales['remise_pct'],
    "montant" : sales['montant'].round(2),
    "montant_net" : (sales['montant']-(sales['montant']*(sales['remise_pct']/100))).round(2),
    "statut" : sales['statut']
})

dim_produit = pd.DataFrame({
    "id_produit" : products['id_produit'],
    "nom_produit" :  products['libelle'],
    "categorie" :  products['categorie'],
    "sous_categorie" :  products['sous_categorie'],
    "cout_achat" : products['cout_achat']
}).drop_duplicates(subset=['id_produit'])


dim_customers = pd.DataFrame({
    "id_client" : customers['id_client'],
    "date_inscription" : customers['date_inscription'],
    "segment" : customers['segment'],
    "ville" : customers['ville'],
    "age" : customers['age']
}).drop_duplicates(subset=['id_client'])

dim_store = pd.DataFrame({
    "id_magasin" : stores['id_magasin'],
    "nom_magasin" : stores['nom_magasin'],
    "ville" : stores['ville'],
    "region" : stores['region'],
    "surface_m2" : stores['surface_m2']
}).drop_duplicates(subset=['id_magasin'])

dimensions = {
    'cle_date': dim_date,
    'id_client': dim_customers,
    'id_produit': dim_produit,
    'id_magasin': dim_store
}

print(fact_sales.head(4),"\n")
print(dim_date.head(2),"\n")
print(dim_produit.head(2),"\n")


             cle_date id_client id_produit id_magasin  quantite  prix_unitaire  remise_pct  montant  montant_net    statut
0 2024-01-18 19:53:00    C02300      P0009        M05         3         160.66         5.0   457.88       434.99  retourne
1 2024-01-08 00:00:00    C02791      P0028        M03         2          70.48         0.0   140.96       140.96     livre
2 2023-04-15 00:00:00    990031      P0017        M04         3         691.12         0.0  2073.36      2073.36     livre
3 2024-11-06 00:00:00    C01883      P0010        M04         4         570.25         0.0  2281.00      2281.00  retourne 

        date  cle_date  annee  mois nom_mois  trimestre  semaine_iso  jour_semaine nom_jour  est_weekend
0 2023-01-01  20230101   2023     1  January          1           52             6   Sunday         True
1 2023-01-02  20230102   2023     1  January          1            1             0   Monday        False 

  id_produit                   nom_produit     categorie       sou

In [32]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

def check_referential_integrity(fact_table, dimensions):
    """dimensions : dict {nom_colonne_cle: dataframe_dimension}"""

    for cle, dimension in dimensions.items():
        orphelines = ~fact_table[cle].isin(dimension[cle])
        nb_orphelines = orphelines.sum()

    if nb_orphelines == 0:
        print("Integritè OK")
    else :
        print({len(nb_orphelines)}," ligne(s) orpheline(s) détectée(s) !")


check_referential_integrity(fact_sales, dimensions)

Integritè OK


---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.